# Stage 5: a self-correcting pricing agent (Self-RAG with LangGraph)

Stage 3's recommender retrieves evidence and cites it, but it trusts whatever the retriever returns and never checks its own answer. This stage turns retrieval and generation into a **LangGraph state machine that grades its own work** and loops back when the work is not good enough, following the Self-RAG pattern:

- **Relevance:** every retrieved document is graded, and irrelevant ones are dropped before the answer is written.
- **Groundedness:** every number in the answer must come from the retrieved facts or the merchant's own figures. If not, the answer is written again.
- **Usefulness:** the answer must give the merchant a price, or say clearly what data is missing. If not, the question is rewritten and retrieval starts again.
- **Query rewriting:** vague merchant questions ("should I drop the price on my sneakers?") are rewritten in the knowledge base's own vocabulary.

A made-up price is a margin loss, so the goal is to catch an unsupported number *before* it reaches the merchant. The last two sections add a retry limit and compare the agent with a plain RAG pipeline on cost and quality.

The graph follows LangGraph's [Self-RAG example](https://github.com/langchain-ai/langgraph/blob/main/examples/rag/langgraph_self_rag.ipynb), based on the [Self-RAG paper](https://arxiv.org/abs/2310.11511).

**Contents:** 1. Knowledge base and retriever · 2. Relevance grader · 3. Generation chain · 4. Groundedness grader · 5. Usefulness grader · 6. Query rewriter · 7. Graph state and nodes · 8. Routing and the compiled graph · 9. Test scenarios · 10. A retry limit · 11. Self-RAG vs naive RAG · Summary

**How this stage is set up**

- **Model:** every LLM call uses `gpt-oss-120b`, on Cerebras when `CEREBRAS_API_KEY` is set in `.env` and on Groq (`openai/gpt-oss-120b`) otherwise.
- **Robust graders:** each grader accepts only a clear `yes` or `no`, and retries the call (up to 3 times) if the model returns anything else.
- **The groundedness grader also sees the merchant's question**, but only as the source of the merchant's own figures. In testing, a correct answer built on the merchant's own $40 cost and $59 competitor price was graded grounded 3 times out of 3 with the question, and *not* grounded 3 times out of 3 without it.
- **The generator checks for category data first,** at medium reasoning effort. An earlier version, at low effort and with a weaker "no data" rule, recommended a grocery margin borrowed from a guideline's worked example, and every grader passed it (see section 4 and the notes after section 9).
- **Every graph run is capped** with LangGraph's `recursion_limit`, and section 10 adds a proper retry limit.
- **The saved outputs come from one full run on Cerebras:** 213 LLM calls, about 196,000 tokens, 2.5 minutes. Answers vary a little from run to run. On Groq's free tier, set `N_RUNS = 1` in section 11 to stay within the daily token limit.

## Setup

Install the dependencies from the repository root (`pip install -r requirements.txt`) and put your API key in a `.env` file at the repository root (copy `.env.example`).

In [1]:
# Import required libraries
from typing import List, Literal
from typing_extensions import TypedDict

from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from pydantic import BaseModel, Field
from langgraph.graph import END, START, StateGraph
from langgraph.errors import GraphRecursionError

import os
import re
import time
import getpass

import pandas as pd

In [2]:
# Set up your API key
# Keys come from the environment or the project's .env file; the notebook only asks if none is found.
# With CEREBRAS_API_KEY set, every LLM call runs on Cerebras (the same open model Groq hosts, with much
# higher rate limits); otherwise on Groq's free tier.
from dotenv import load_dotenv

load_dotenv()
if os.environ.get("CEREBRAS_API_KEY"):
    PROVIDER, MODEL = "Cerebras", "gpt-oss-120b"
else:
    PROVIDER, MODEL = "Groq", "openai/gpt-oss-120b"
    if not os.environ.get("GROQ_API_KEY"):
        print("Please enter your Groq API key:")
        print("(You can get one free at: https://console.groq.com/)")
        os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API Key: ")


class CallCounter(BaseCallbackHandler):
    """Counts every LLM call and the tokens it used (for the cost comparison in section 11)."""

    def __init__(self):
        self.calls = 0
        self.tokens = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.calls += 1

    def on_llm_end(self, response, **kwargs):
        for generations in response.generations:
            for generation in generations:
                usage = getattr(getattr(generation, "message", None), "usage_metadata", None) or {}
                self.tokens += usage.get("total_tokens", 0)

    def snapshot(self):
        return self.calls, self.tokens


COUNTER = CallCounter()


def make_llm(max_tokens: int = 1024, reasoning_effort: str = "low"):
    """gpt-oss-120b at temperature 0 (Cerebras if its key is set, else Groq).

    max_tokens: gpt-oss spends part of this budget on hidden reasoning before it answers, so it must
        leave room for both; without a limit Groq can cut a reply off and return nothing.
    reasoning_effort: how much hidden reasoning the model does ("low" is enough for short answers
        and yes/no grading).
    max_retries: wait and retry when a rate limit is hit, instead of failing the run.
    """
    settings = dict(temperature=0, max_tokens=max_tokens, reasoning_effort=reasoning_effort,
                    max_retries=10, callbacks=[COUNTER])
    if PROVIDER == "Cerebras":
        return ChatOpenAI(model=MODEL, base_url="https://api.cerebras.ai/v1",
                          api_key=os.environ["CEREBRAS_API_KEY"], **settings)
    return ChatGroq(model=MODEL, **settings)


print(f"API key configured! Using {PROVIDER}: {MODEL}")

API key configured! Using Cerebras: gpt-oss-120b


## The pricing knowledge base

Twenty short entries written for this stage (fictional, in $): category elasticity benchmarks, historical margins, past pricing decisions with their outcomes, and general pricing guidelines. Stage 3's knowledge base is larger and in ₹; this smaller one keeps every retrieval and grading step easy to follow in the trace.

Two properties matter:
1. **Electronics, Footwear, Apparel and Luxury goods are well covered.**
2. **Some things a merchant might ask about, such as grocery or SaaS seat pricing, are not covered at all.** A plain RAG agent may borrow a figure from another category. This agent has to notice the gap and say so.

In [3]:
# Pricing knowledge base: (text, metadata) pairs
pricing_knowledge = [
    # --- Category elasticity benchmarks ---
    ("Category elasticity benchmark: Electronics. Price elasticity is HIGH (elasticity coefficient "
     "-1.8 to -2.4). A 10% price increase typically reduces unit volume by 18-24%. Consumers "
     "comparison-shop heavily and switching cost is near zero. Recommended strategy: price at or "
     "slightly below the leading competitor and win on volume.", {"type": "elasticity", "category": "Electronics"}),

    ("Category elasticity benchmark: Footwear. Price elasticity is HIGH for mass-market brands "
     "(-1.5 to -2.0) but MEDIUM for established athletic brands with loyalty (-0.9 to -1.3). "
     "Seasonal discounting is expected by the market in Q1 and Q3.", {"type": "elasticity", "category": "Footwear"}),

    ("Category elasticity benchmark: Apparel. Price elasticity is MEDIUM (-1.0 to -1.4). Demand is "
     "driven more by seasonality and trend than by absolute price. Full-price sell-through rarely "
     "exceeds 60%; markdown planning is part of the pricing decision.", {"type": "elasticity", "category": "Apparel"}),

    ("Category elasticity benchmark: Luxury Goods. Price elasticity is LOW (-0.3 to -0.6) and can be "
     "positive (Veblen effect) for aspirational brands. Discounting damages long-term brand equity. "
     "Recommended strategy: hold price, never discount below 10%.", {"type": "elasticity", "category": "Luxury"}),

    # --- Historical margins ---
    ("Historical margin data: Electronics. Average realised gross margin is 12-18%. Top quartile "
     "retailers achieve 21% by attaching warranties and accessories rather than by raising unit price. "
     "Margins below 8% are unsustainable after returns and payment processing.",
     {"type": "margin", "category": "Electronics"}),

    ("Historical margin data: Footwear. Average realised gross margin is 38-45%. Athletic footwear "
     "sits at the top of that band. After markdowns, realised margin typically lands 6-9 points below "
     "the list-price margin.", {"type": "margin", "category": "Footwear"}),

    ("Historical margin data: Apparel. Average realised gross margin is 45-55% at list price, falling "
     "to 30-36% after seasonal markdowns. Plan the initial price assuming roughly 40% of units sell "
     "at a discount.", {"type": "margin", "category": "Apparel"}),

    ("Historical margin data: Luxury Goods. Average realised gross margin is 60-70%. Markdown activity "
     "is minimal; less than 10% of units are ever discounted.",
     {"type": "margin", "category": "Luxury"}),

    # --- Past pricing decisions ---
    ("Past pricing decision: Samsung Galaxy Smartphone. Category Electronics. Cost $450, competitor "
     "priced at $529, we recommended $499 for an 11% margin. Outcome: SUCCESSFUL — gained 15% market "
     "share in the quarter. Reasoning: aggressive pricing in a high-elasticity category drove volume "
     "and the accessory attach rate recovered the thin unit margin.",
     {"type": "decision", "category": "Electronics"}),

    ("Past pricing decision: Sony Wireless Headphones. Category Electronics. Cost $80, competitor "
     "priced at $149, we recommended $139 for a 42% margin. Outcome: MIXED — margin target met but "
     "volume fell 12% below forecast. Reasoning: we underestimated elasticity; $129 would likely have "
     "maximised total gross profit.", {"type": "decision", "category": "Electronics"}),

    ("Past pricing decision: Puma Running Sneakers. Category Footwear. Cost $1800 (INR-denominated "
     "cost base), competitor priced at $2799, we recommended $2599 for a 31% margin. Outcome: "
     "SUCCESSFUL — hit the 30% target margin while undercutting the competitor by 7%. Reasoning: high "
     "elasticity meant the 7% gap was enough to shift preference without starting a price war.",
     {"type": "decision", "category": "Footwear"}),

    ("Past pricing decision: Nike Court Sneakers. Category Footwear. Cost $2100, competitor priced at "
     "$3199, we recommended $3149 for a 33% margin. Outcome: SUCCESSFUL — brand loyalty meant near "
     "parity pricing held volume. Reasoning: established athletic brands behave as medium-elasticity, "
     "not high-elasticity, so matching the competitor was safe.",
     {"type": "decision", "category": "Footwear"}),

    ("Past pricing decision: Winter Jacket, own label. Category Apparel. Cost $60, competitor priced "
     "at $159, we recommended $149 for a 60% list margin. Outcome: MIXED — 55% of units sold at "
     "markdown, realised margin was 34%. Reasoning: the list price was correct but the markdown "
     "cadence started too early in the season.", {"type": "decision", "category": "Apparel"}),

    ("Past pricing decision: Designer Leather Handbag. Category Luxury. Cost $400, competitor priced "
     "at $1450, we recommended $1499 for a 73% margin. Outcome: SUCCESSFUL — pricing above the "
     "competitor reinforced premium positioning and volume was unaffected. Reasoning: low elasticity "
     "plus Veblen dynamics reward a price premium.", {"type": "decision", "category": "Luxury"}),

    ("Past pricing decision: Cotton T-Shirt multipack. Category Apparel. Cost $12, competitor priced "
     "at $29, we recommended $27 for a 56% margin. Outcome: SUCCESSFUL — commodity apparel is price "
     "led, and a small visible discount drove basket attachment.",
     {"type": "decision", "category": "Apparel"}),

    # --- Pricing guidelines ---
    ("Pricing guideline: Margin floor rule. Never recommend a price below cost / (1 - minimum "
     "acceptable margin). For a 25% minimum margin and a $400 cost, the floor price is $533.33. "
     "Recommending below the floor requires an explicit loss-leader justification.",
     {"type": "guideline", "category": "General"}),

    ("Pricing guideline: High elasticity rule. In HIGH elasticity categories, price within 0-8% below "
     "the leading competitor. Undercutting by more than 15% invites retaliation and a price war that "
     "destroys category margin for everyone.", {"type": "guideline", "category": "General"}),

    ("Pricing guideline: Low elasticity rule. In LOW elasticity categories, price at or above the "
     "competitor. Discounting is value-destructive because volume will not respond enough to "
     "compensate for the lost margin.", {"type": "guideline", "category": "General"}),

    ("Pricing guideline: Target margin reconciliation. When the target margin price and the "
     "competitive price conflict, state both numbers explicitly, recommend one, and name the "
     "trade-off being accepted. Never silently pick one and hide the other.",
     {"type": "guideline", "category": "General"}),

    ("Pricing guideline: Insufficient data rule. If the knowledge base contains no elasticity "
     "benchmark or margin history for the requested category, say so explicitly and do not "
     "extrapolate from an unrelated category.", {"type": "guideline", "category": "General"}),
]

print(f"Knowledge base entries: {len(pricing_knowledge)}")
print(f"Categories covered: {sorted(set(m['category'] for _, m in pricing_knowledge))}")

Knowledge base entries: 20
Categories covered: ['Apparel', 'Electronics', 'Footwear', 'General', 'Luxury']


## 1. Knowledge base and retriever

Each entry becomes one `Document`, with its type and category as metadata. The entries are embedded locally with `all-MiniLM-L6-v2` and stored in Chroma, and the retriever returns the 4 closest.

**No chunking:** each entry is already one self-contained idea, so a grader can judge it as a whole. A chunk cut from the middle of an entry could be half relevant.

In [4]:
# 1. Documents, embeddings, vector store and retriever

# Step 1: one Document per knowledge-base entry (each entry is already one self-contained idea)
docs = [Document(page_content=text, metadata=meta) for text, meta in pricing_knowledge]

# Step 2: local HuggingFace embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={"device": "cpu"})

# Step 3: the Chroma vector store. Chroma keeps in-memory collections for the whole Python session,
# so delete any earlier copy first; otherwise re-running this cell would index every entry twice.
Chroma(collection_name="pricing-self-rag", embedding_function=embeddings).delete_collection()
vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings,
                                    collection_name="pricing-self-rag")

# Step 4: the retriever returns the 4 closest entries
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


def label(doc):
    """Short tag for a document, e.g. 'margin / Footwear'."""
    return f"{doc.metadata['type']} / {doc.metadata['category']}"


# Step 5: sanity check
print(f"Indexed {len(docs)} documents.\n")
print("Retrieved for 'footwear sneakers margin':")
for i, doc in enumerate(retriever.invoke("footwear sneakers margin"), 1):
    print(f"\n{i}. [{label(doc)}]\n   {doc.page_content}")

print()
print("Retriever ready!" if retriever else "Retriever not yet implemented.")

Indexed 20 documents.

Retrieved for 'footwear sneakers margin':

1. [margin / Footwear]
   Historical margin data: Footwear. Average realised gross margin is 38-45%. Athletic footwear sits at the top of that band. After markdowns, realised margin typically lands 6-9 points below the list-price margin.

2. [decision / Footwear]
   Past pricing decision: Puma Running Sneakers. Category Footwear. Cost $1800 (INR-denominated cost base), competitor priced at $2799, we recommended $2599 for a 31% margin. Outcome: SUCCESSFUL — hit the 30% target margin while undercutting the competitor by 7%. Reasoning: high elasticity meant the 7% gap was enough to shift preference without starting a price war.

3. [decision / Footwear]
   Past pricing decision: Nike Court Sneakers. Category Footwear. Cost $2100, competitor priced at $3199, we recommended $3149 for a 33% margin. Outcome: SUCCESSFUL — brand loyalty meant near parity pricing held volume. Reasoning: established athletic brands behave as medium

## 2. Relevance grader

The first self-correction step. For **each** retrieved document, a small structured LLM call answers `yes` or `no`: is this document relevant to the merchant's question? Irrelevant documents are dropped before they reach the generator, so, for example, a Luxury margin figure can't leak into a sneaker recommendation.

The test is deliberately lenient: the aim is to remove wrong retrievals, not to demand a perfect match. Each grader accepts only a clear `yes` or `no`, and retries the call (up to 3 times) if the model returns anything else.

In [5]:
# 2. Relevance grader

# Step 1: Data model
class GradeDocuments(BaseModel):
    """Binary score for relevance check on retrieved pricing documents."""
    binary_score: str = Field(description="Document is relevant to the pricing question, 'yes' or 'no'")


def check_score(result):
    """Accept only a clear 'yes' or 'no' (any case). Anything else raises, so the call is retried."""
    score = (getattr(result, "binary_score", None) or "").strip().strip(".'\"").lower()
    if score not in ("yes", "no"):
        raise ValueError(f"grader did not answer yes or no: {result!r}")
    result.binary_score = score
    return result


# Step 2 & 3: Grader LLM with structured output (a short yes/no needs little reasoning)
llm_grader = make_llm(max_tokens=1024)
structured_llm_grader = llm_grader.with_structured_output(GradeDocuments)

# Step 4: Grading prompt
system = """You are a grader assessing the relevance of a document retrieved from a retail pricing
knowledge base to a merchant's pricing question. This is not a stringent test: the goal is only to
filter out erroneous retrievals.

Grade the document relevant ('yes') if any of these is true:
- it is about the same product category as the question;
- it covers a pricing concept the question needs (price elasticity, margin, competitor positioning,
  discounting) in a way that applies to the question's category, for example a general pricing
  guideline that is not tied to one category;
- it is a past pricing decision for a comparable product.
Grade it not relevant ('no') if it is about a different, specific category and offers nothing that
applies to the question: a Luxury margin figure does not help price sneakers just because both
mention margins.

Give a binary score 'yes' or 'no'."""
grade_prompt = ChatPromptTemplate.from_messages([
    ("system", system),
    ("human", "Retrieved document: \n\n {document} \n\n Pricing question: {question}"),
])

# Step 5: Compose the grader (plus the yes/no check, retried up to 3 times)
retrieval_grader = (grade_prompt | structured_llm_grader | RunnableLambda(check_score)).with_retry(
    stop_after_attempt=3)

# Step 6: Test on one relevant and one irrelevant document
test_question = "What margin should I target on Puma sneakers costing $1800?"


def find_doc(doc_type, category):
    return next(d for d in docs if d.metadata == {"type": doc_type, "category": category})


print(f"Question: {test_question}\n")
for doc, expected in [(find_doc("margin", "Footwear"), "yes"),
                      (find_doc("elasticity", "Luxury"), "no"),
                      (find_doc("margin", "Luxury"), "no")]:  # extra: same concept, wrong category
    score = retrieval_grader.invoke({"question": test_question, "document": doc.page_content}).binary_score
    print(f"  {label(doc):22} -> {score:3}  (expected {expected})")

# The four documents the retriever actually returns for this question
print("\nGrading what the retriever returns for the question:")
test_docs = retriever.invoke(test_question)
for doc in test_docs:
    score = retrieval_grader.invoke({"question": test_question, "document": doc.page_content}).binary_score
    print(f"  {label(doc):22} -> {score}")

Question: What margin should I target on Puma sneakers costing $1800?

  margin / Footwear      -> yes  (expected yes)
  elasticity / Luxury    -> no   (expected no)
  margin / Luxury        -> no   (expected no)

Grading what the retriever returns for the question:
  decision / Footwear    -> yes
  decision / Footwear    -> yes
  guideline / General    -> yes
  margin / Footwear      -> yes


**What this shows**

- **The grader drops the right documents.** The Footwear margin entry passes, and both Luxury entries are rejected, including the Luxury *margin* entry, which shares the word "margin" with the question. All four documents retrieved for the Puma question pass.
- **If the Luxury entry had stayed in,** the generator would see "60-70% gross margin" and "hold price, never discount" next to the footwear data. It could then quote the wrong margin band, or follow luxury advice (price above the competitor) that contradicts the footwear evidence (undercut by about 7%).
- **Why one small call per document:**
  - each verdict is independent, so one confusing document can't affect the grades of the others;
  - each reply is a single yes/no field, which is easy to check and retry, whereas one bad parse of a combined reply would lose every verdict;
  - the trace shows exactly which document was dropped, and the calls could run in parallel.

## 3. Generation chain

The answer writer: a pricing-analyst prompt that may use only the retrieved context. It must give one price with its margin and cite its evidence. If the context has no data for the product's category, it must say "The knowledge base does not contain data for this category". Most of the quality control lives in the graders around it.

In [6]:
# 3. Pricing generation chain

# Step 1: Generation LLM. Medium reasoning effort: at "low", the model kept skipping the "no data for
# this category" check below (see the notes after the test scenarios). max_tokens is 4096 because
# gpt-oss's hidden reasoning shares that budget with the answer.
llm = make_llm(max_tokens=4096, reasoning_effort="medium")

# Step 2: Pricing prompt with {context} and {question}
pricing_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a pricing analyst for a retail merchant. Answer the merchant's pricing
question using ONLY the retrieved context below, never your own pricing intuition or outside market
knowledge.

Your answer must:
1. State one specific recommended price and the resulting gross margin, where
   margin = (price - cost) / price. Show that calculation.
2. Cite the retrieved evidence that supports the price by name (elasticity benchmark, margin
   history, past decision or guideline).
3. If the target-margin price and the competitive price conflict, state both numbers, recommend
   one and name the trade-off you accept.

If the merchant has not given the numbers a specific price needs (their cost, the competitor's
price), say which numbers are missing and give the rule from the context they should apply instead.
Before answering, check whether the context has any data for the product's own category: an
elasticity benchmark, margin history or past pricing decision for that category. General pricing
guidelines are not category data. If there is no category data, start your answer with "The
knowledge base does not contain data for this category", do not recommend a target margin or a
price, and do not borrow figures from another category or from a guideline's worked example (those
numbers only illustrate the rule). You may still explain a general guideline as a rule the merchant
can apply with their own numbers.
Write prices with the currency symbol the merchant uses. Keep the answer under 200 words."""),
    ("human", "Retrieved context:\n\n{context}\n\nMerchant question: {question}"),
])


# Step 3: Format helper
def format_docs(docs):
    """Join retrieved documents into a single context string."""
    return "\n\n".join(doc.page_content for doc in docs)


def one_copy(text):
    """gpt-oss sometimes sends its whole reply twice in a row ("abcabc"); keep one copy."""
    text = text.strip()
    half = len(text) // 2
    for cut in range(half - 2, half + 3):
        if 0 < cut < len(text) and text[:cut].strip() == text[cut:].strip():
            return text[:cut].strip()
    return text


# Step 4: Build the chain
rag_chain = pricing_prompt | llm | StrOutputParser() | RunnableLambda(one_copy)

# Step 5: Test the chain on the footwear question and the documents retrieved in section 2
generation = rag_chain.invoke({"context": format_docs(test_docs), "question": test_question})
print(generation)

**Recommended price:** $2599  
**Resulting gross margin:** \((2599‑1800) ÷ 2599 = 799 ÷ 2599 ≈ 0.308\) → **30.8 %** (rounded to 31 %).

**Evidence:** The past pricing decision for *Puma Running Sneakers* (Footwear) used a cost of $1800, a competitor price of $2799, and recommended $2599, achieving a 31 % margin and meeting the 30 % target margin (see “Past pricing decision: Puma Running Sneakers”).  

Because the recommended price undercuts the competitor by about 7 % and the elasticity was judged high, this margin aligns with the successful outcome previously recorded. No conflict with a competing price exists, so the 31 % margin is the advised target.


## 4. Groundedness grader

The most important check in a pricing agent: is every number in the recommendation traceable to the retrieved facts?

- The grader gets the documents (the evidence) and the merchant's question, but may use the question only for the merchant's own figures: their cost, the competitor's price they quote, their target margin.
- Numbers derived by arithmetic, such as a floor price computed from a cost and a margin, count as grounded.
- When this grader says `no`, the graph writes the answer again.

It is tested on the real answer from section 3, on a fabricated answer, and on a subtler case from an earlier test run.

In [7]:
# 4. Groundedness (hallucination) grader

# Step 1: Data model
class GradeHallucinations(BaseModel):
    """Binary score for whether the pricing recommendation is grounded in retrieved facts."""
    binary_score: str = Field(description="Pricing recommendation is grounded in the retrieved facts, 'yes' or 'no'")


# Step 2 & 3: Structured grader + pricing-specific system prompt. Checking arithmetic needs more
# thought than a relevance check, so this grader gets medium reasoning effort.
structured_llm_hallucination = make_llm(max_tokens=4096, reasoning_effort="medium").with_structured_output(
    GradeHallucinations)

system = """You are a grader checking whether a pricing recommendation is grounded in a set of
retrieved pricing facts. Check every number and claim in the recommendation:
- Every margin percentage must appear in the facts or follow arithmetically from numbers that do
  (margin = (price - cost) / price).
- Every competitor price, cost and historical outcome must appear in the facts.
- Every elasticity claim must match the benchmark in the facts.
- The merchant's own figures (their cost, the competitor's price they quote, their target margin)
  may come from the merchant's question instead of the facts. The question is NOT evidence for
  anything else: benchmarks, margin history and past outcomes must come from the facts.
- The figures behind the recommendation must apply to the product being priced. A number in a
  guideline's worked example (an example cost, an example minimum margin) only illustrates the
  rule: basing the recommendation on it, as if it were the merchant's own target or a benchmark
  for their product, is NOT grounded. Likewise, a benchmark, margin or past decision from one
  product category is NOT grounded evidence for a product in a different category.
Arithmetic derived from given numbers (for example a floor price computed from a cost and a margin,
or the margin a given price produces) counts as grounded: this is not a test of whether numbers were
copied word for word. Small rounding differences are fine. Saying that the facts do not contain data
for something is grounded.
Answer 'no' if any number or claim is not supported; otherwise answer 'yes'."""
hallucination_prompt = ChatPromptTemplate.from_messages([
    ("system", system),
    ("human", "Set of pricing facts: \n\n {documents} \n\n Merchant question: {question} \n\n "
              "Pricing recommendation: {generation}"),
])

# Step 4: Compose
hallucination_grader = (hallucination_prompt | structured_llm_hallucination
                        | RunnableLambda(check_score)).with_retry(stop_after_attempt=3)

# Step 5: Test on the real generation AND on a fabricated one
fabricated_generation = (
    "Recommend $2,999 for the Puma sneakers. Footwear margins historically run at 85% "
    "and the competitor sells at $4,500."
)
facts = format_docs(test_docs)
for name, gen, expected in [("real answer (section 3)", generation, "yes"),
                            ("fabricated generation", fabricated_generation, "no")]:
    score = hallucination_grader.invoke(
        {"documents": facts, "question": test_question, "generation": gen}).binary_score
    print(f"{name:26} -> grounded: {score:3}  (expected {expected})")

# A subtler case, taken from an earlier test run of this agent (see the notes below): for a
# category the knowledge base doesn't cover, it took the 25% from the margin floor rule's worked
# example and recommended it as the merchant's target margin. Graded 3 times, because the verdict
# on this case changes from call to call.
grocery_question = "What margin should I target on organic grocery produce costing $3 per unit?"
borrowed_generation = (
    "Using the margin floor rule with a minimum acceptable margin of 25%: floor price = $3 / (1 - 0.25) "
    "= $4.00. Recommended price: $4.00, a 25% gross margin, so target a 25% margin on the produce."
)
scores = [hallucination_grader.invoke({"documents": find_doc("guideline", "General").page_content,
                                       "question": grocery_question,
                                       "generation": borrowed_generation}).binary_score
          for _ in range(3)]
print(f"{'example number reused':26} -> grounded: {', '.join(scores)}  (expected no)")

real answer (section 3)    -> grounded: yes  (expected yes)
fabricated generation      -> grounded: no   (expected no)
example number reused      -> grounded: yes, yes, yes  (expected no)


**What this shows**

- **It works on clear cases:** the real answer is accepted, and the fabricated one (an 85% margin and a $4,500 competitor price) is rejected.
- **It misses the subtle case:** the reused example number passed all 3 times in this run, even though the prompt now describes exactly this mistake.
  - In an earlier version of this notebook, the agent's own grocery answer was "$4.00, a 25% margin", with the 25% taken from the margin floor rule's worked example ("for a 25% minimum margin and a $400 cost…").
  - Across earlier runs, the verdict on this case changed from call to call.
  - Asking the grader to list the unsupported claims before its verdict made things worse: it then rejected a correct Puma answer.

  So this grader is a safety net, not a guarantee. The fix that worked went into the generator prompt (section 3).
- **Why the grader gets the documents, not just the question:** groundedness asks "is every claim backed by evidence?", and the evidence is the documents. The question contains no evidence, so a grader that saw only the question could check relevance but not truth. The question is passed as well, but only for the merchant's own figures (see the setup notes).

## 5. Usefulness grader

Grounded is not the same as useful: "pricing depends on many factors" is perfectly grounded and worthless to a merchant. This grader checks that the answer gives a specific price, or says explicitly what data is missing, with reasoning tied to the merchant's numbers.

When it says `no`, the graph rewrites the **question** and retrieves again, because the problem is upstream.

In [8]:
# 5. Usefulness (answer) grader

# Step 1: Data model GradeAnswer with binary_score
class GradeAnswer(BaseModel):
    """Binary score for whether the recommendation resolves the merchant's question."""
    binary_score: str = Field(description="Recommendation resolves the merchant's pricing question, 'yes' or 'no'")


# Step 2: System prompt - what makes a pricing answer USEFUL?
system = """You are a grader checking whether a pricing recommendation is useful to the merchant
who asked the question. A useful answer does both of these:
1. It gives a specific price, OR it says explicitly, with the reason, that the data needed for a
   specific price is missing (from the knowledge base or from the merchant's question) and tells
   the merchant what to do instead.
2. It explains the reasoning that connects the recommendation to the merchant's stated constraints
   (cost, competitor price, target margin), where the merchant gave any.
Answer 'no' for vague answers ("it depends on many factors"), answers that dodge the question, and
bare numbers with no reasoning. You are not checking whether the numbers are correct.
Answer 'yes' or 'no'."""
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", system),
    ("human", "Merchant question: \n\n {question} \n\n Pricing recommendation: {generation}"),
])

structured_llm_answer = make_llm(max_tokens=1024).with_structured_output(GradeAnswer)
answer_grader = (answer_prompt | structured_llm_answer | RunnableLambda(check_score)).with_retry(
    stop_after_attempt=3)

# Step 4: Test on the real generation and on a vague non-answer
vague_generation = "Pricing is a complex decision that depends on many market factors."
for name, gen, expected in [("real answer (section 3)", generation, "yes"),
                            ("vague non-answer", vague_generation, "no")]:
    score = answer_grader.invoke({"question": test_question, "generation": gen}).binary_score
    print(f"{name:26} -> useful: {score:3}  (expected {expected})")

real answer (section 3)    -> useful: yes  (expected yes)
vague non-answer           -> useful: no   (expected no)


## 6. Query rewriter

Merchants don't ask retrieval-friendly questions. The rewriter turns "my sneakers aren't selling, should I drop the price?" into the knowledge base's vocabulary (category, elasticity, margin, markdown, competitor positioning), keeping every number the merchant gave. It is the same idea as HyDE in stage 2, used inside a loop.

The cell also compares what the original and the rewritten question retrieve.

In [9]:
# 6. Query rewriter

# Step 1: Rewriter LLM (temperature=0, set in make_llm)
rewriter_llm = make_llm(max_tokens=1024)

# Step 2: System prompt for pricing-aware query rewriting
system = """You rewrite a merchant's pricing question into a better query for vector-store
retrieval over a retail pricing knowledge base. The knowledge base holds category elasticity
benchmarks, historical gross margins by category, past pricing decisions with their outcomes, and
pricing guidelines.
Reason about what the merchant actually wants to know, then write one query that names:
- the product category (for example Electronics, Footwear, Apparel, Luxury Goods);
- the pricing concepts the question implies (price elasticity, gross margin, markdown or discount,
  competitor price positioning, past pricing decisions).
Keep every number and constraint the merchant gave (cost, competitor price, target margin, product)
and do not invent new ones. Reply with the rewritten query only."""
re_write_prompt = ChatPromptTemplate.from_messages([
    ("system", system),
    ("human", "Here is the initial merchant question: \n\n {question} \n Formulate an improved question."),
])

# Step 3: Compose
question_rewriter = re_write_prompt | rewriter_llm | StrOutputParser() | RunnableLambda(one_copy)

# Step 4: Test the rewriter
vague_merchant_query = "my sneakers aren't selling, should I drop the price?"
rewritten_query = question_rewriter.invoke({"question": vague_merchant_query})
print(f"Original : {vague_merchant_query}")
print(f"Rewritten: {rewritten_query}")

# Does the rewrite change what gets retrieved?
for name, query in [("original", vague_merchant_query), ("rewritten", rewritten_query)]:
    print(f"\nTop 4 for the {name} query:")
    for doc in retriever.invoke(query):
        print(f"  [{label(doc):22}] {doc.page_content[:80]}...")

Original : my sneakers aren't selling, should I drop the price?
Rewritten: Footwear sneakers: Given low sales, what does the pricing knowledge base suggest about price elasticity, expected gross margin impact, and optimal markdown or discount level (including competitor price positioning and past pricing decisions) for this product?

Top 4 for the original query:
  [decision / Footwear   ] Past pricing decision: Nike Court Sneakers. Category Footwear. Cost $2100, compe...
  [decision / Footwear   ] Past pricing decision: Puma Running Sneakers. Category Footwear. Cost $1800 (INR...
  [decision / Apparel    ] Past pricing decision: Winter Jacket, own label. Category Apparel. Cost $60, com...
  [decision / Apparel    ] Past pricing decision: Cotton T-Shirt multipack. Category Apparel. Cost $12, com...

Top 4 for the rewritten query:
  [elasticity / Footwear ] Category elasticity benchmark: Footwear. Price elasticity is HIGH for mass-marke...
  [decision / Footwear   ] Past pricing decisio

**What this shows**

- **Original:** "my sneakers aren't selling, should I drop the price?"
- **Rewritten:** "Footwear sneakers: Given low sales, what does the pricing knowledge base suggest about price elasticity, expected gross margin impact, and optimal markdown or discount level (including competitor price positioning and past pricing decisions) for this product?"

**Vocabulary added:** the category (*Footwear*), *price elasticity*, *gross margin*, *markdown or discount level*, *competitor price positioning* and *past pricing decisions*.

**Retrieval changes:**
- The original question retrieves two Footwear decisions plus two **Apparel** decisions (a winter jacket and a T-shirt multipack). They match "not selling / drop the price" through their markdown wording.
- The rewritten question retrieves the **Footwear elasticity benchmark**, the Nike decision and the **high and low elasticity guidelines**, with no Apparel at all.

## 7. Graph state and nodes

The state is a dict that flows between nodes; each node returns a partial update to it.

| Node | Does | Returns |
|---|---|---|
| `retrieve` | retrieves from the vector store | `documents`, `question` |
| `grade_documents` | grades each document, keeps only the relevant ones | `documents` (filtered), `question` |
| `generate` | writes the recommendation with `rag_chain` | `documents`, `question`, `generation` |
| `transform_query` | rewrites the question | `documents`, `question` (rewritten) |

Each node prints a trace line (`---RETRIEVE---` and so on), so every run shows the path it took.

In [10]:
# 7a. Graph state

class GraphState(TypedDict):
    """
    Represents the state of our pricing agent graph.

    Attributes:
        question: the merchant's pricing question (rewritten by transform_query)
        generation: the LLM's price recommendation
        documents: list of retrieved pricing documents (Document objects)
    """
    question: str
    generation: str
    documents: List[Document]

In [11]:
# 7b. The four nodes

def retrieve(state):
    """Retrieve pricing documents for the question."""
    print("---RETRIEVE---")
    question = state["question"]
    documents = retriever.invoke(question)
    return {"documents": documents, "question": question}


def generate(state):
    """Generate a price recommendation from the retrieved documents."""
    print("---GENERATE PRICE RECOMMENDATION---")
    question = state["question"]
    documents = state["documents"]
    generation = rag_chain.invoke({"context": format_docs(documents), "question": question})
    return {"documents": documents, "question": question, "generation": generation}


def grade_documents(state):
    """Keep only the pricing documents graded relevant to the question."""
    print("---CHECK DOCUMENT RELEVANCE TO PRICING QUESTION---")
    question = state["question"]
    filtered_docs = []
    for d in state["documents"]:
        score = retrieval_grader.invoke({"question": question, "document": d.page_content})
        if score.binary_score == "yes":
            print(f"---GRADE: PRICING DOCUMENT RELEVANT---      [{label(d)}]")
            filtered_docs.append(d)
        else:
            print(f"---GRADE: PRICING DOCUMENT NOT RELEVANT---  [{label(d)}]")
    return {"documents": filtered_docs, "question": question}


def transform_query(state):
    """Rewrite the merchant question into a retrieval-optimised pricing question."""
    print("---TRANSFORM QUERY---")
    question = state["question"]
    better_question = question_rewriter.invoke({"question": question})
    print(f"   new question: {better_question}")
    return {"documents": state["documents"], "question": better_question}


print("Nodes defined!")

Nodes defined!


## 8. Routing and the compiled graph

Two routing functions decide where to go next:
- after `grade_documents`: if no document survived, rewrite the question; otherwise write the answer;
- after `generate`: check groundedness, then usefulness.

```
START → retrieve → grade_documents → [decide_to_generate]
                                       ├─ "transform_query" → transform_query → retrieve
                                       └─ "generate" → generate → [grade_generation...]
                                                                    ├─ "not supported" → generate
                                                                    ├─ "not useful"    → transform_query
                                                                    └─ "useful"        → END
```

There are two loops:
- **not supported → write again:** the retrieval was fine, but the answer used numbers it can't back up;
- **not useful → retrieve again:** the answer was fine for what was retrieved, but the retrieval missed what the merchant needed.

In [12]:
# 8a & 8b. The two routing functions

def decide_to_generate(state):
    """Decide whether to generate an answer or rewrite the query."""
    print("---ASSESS GRADED PRICING DOCUMENTS---")
    if not state["documents"]:
        print("---DECISION: NO RELEVANT PRICING DOCUMENTS, TRANSFORM QUERY---")
        return "transform_query"
    print("---DECISION: GENERATE---")
    return "generate"


def grade_generation_v_documents_and_question(state):
    """Check groundedness, then usefulness, and route accordingly."""
    print("---CHECK HALLUCINATIONS IN PRICE RECOMMENDATION---")
    question = state["question"]
    documents = state["documents"]
    generation = state["generation"]

    score = hallucination_grader.invoke(
        {"documents": format_docs(documents), "question": question, "generation": generation})
    if score.binary_score == "yes":
        print("---DECISION: RECOMMENDATION IS GROUNDED IN PRICING DATA---")
        print("---GRADE RECOMMENDATION VS QUESTION---")
        score = answer_grader.invoke({"question": question, "generation": generation})
        if score.binary_score == "yes":
            print("---DECISION: RECOMMENDATION ANSWERS THE PRICING QUESTION---")
            return "useful"
        print("---DECISION: RECOMMENDATION DOES NOT ANSWER THE PRICING QUESTION---")
        return "not useful"
    print("---DECISION: RECOMMENDATION NOT GROUNDED, RE-TRY---")
    return "not supported"


print("Edge functions defined!")

Edge functions defined!


In [13]:
# 8c. Build and compile the graph

workflow = StateGraph(GraphState)

# Step 1: Add the four nodes
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("generate", generate)
workflow.add_node("transform_query", transform_query)

# Step 2: Add the fixed edges
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_edge("transform_query", "retrieve")

# Step 3: Add the conditional edge from grade_documents
workflow.add_conditional_edges(
    "grade_documents",
    decide_to_generate,
    {"transform_query": "transform_query", "generate": "generate"},
)

# Step 4: Add the conditional edge from generate
workflow.add_conditional_edges(
    "generate",
    grade_generation_v_documents_and_question,
    {"not supported": "generate", "useful": END, "not useful": "transform_query"},
)

# Step 5: Compile
app = workflow.compile()

print("Graph compiled!" if app else "Graph not yet compiled.")

Graph compiled!


In [14]:
# Visualise the graph
# draw_mermaid_png() needs an internet service and draw_ascii() needs the grandalf package, so this
# prints the edges as text instead (dotted = conditional edge, labelled with the routing value).
def print_edges(graph):
    for edge in graph.get_graph().edges:
        if edge.conditional:
            print(f"  {edge.source:20} ..{edge.data or edge.target}..> {edge.target}")
        else:
            print(f"  {edge.source:20} ------> {edge.target}")


print_edges(app)

  __start__            ------> retrieve
  generate             ..useful..> __end__
  generate             ..not supported..> generate
  generate             ..not useful..> transform_query
  grade_documents      ..generate..> generate
  grade_documents      ..transform_query..> transform_query
  retrieve             ------> grade_documents
  transform_query      ------> retrieve


## 9. Test scenarios

Four questions, each streamed through the graph with its full trace:

- **A, clean path:** a well-covered question (Puma sneakers, with cost, competitor price and target margin).
- **B, vague question:** "my sneakers aren't selling, should I drop the price?", with no numbers at all.
- **C, out of scope:** organic grocery produce, a category the knowledge base doesn't cover. The agent must say so, not invent a grocery margin.
- **D, rewrite needed:** a phone case, where every document from the first retrieval fails grading.

**Helper for the scenarios.** `run_agent()` streams the graph and prints each node as it finishes (the
nodes print their own `---TRACE---` lines). It returns the final state and a summary of the run:
the path through the graph, how many times the query was rewritten, the number of LLM calls and
tokens, and the time taken. Every run passes LangGraph's `recursion_limit`, so a loop that never
ends stops with a message instead of running up API calls.

In [15]:
def run_agent(graph, question, recursion_limit=25, verbose=True):
    """Stream the graph on one question. Returns (final_state, run_summary)."""
    calls_before, tokens_before = COUNTER.snapshot()
    started = time.time()
    path, state = [], {"question": question}
    finished = True
    try:
        for output in graph.stream({"question": question}, config={"recursion_limit": recursion_limit}):
            for key, value in output.items():
                if verbose:
                    print(f"Node '{key}':")
                    print("\n---\n")
                path.append(key)
                state.update(value or {})
    except GraphRecursionError:
        finished = False
        print(f"!!! Stopped by LangGraph's recursion limit ({recursion_limit} steps): the loop never ended.")
    calls_after, tokens_after = COUNTER.snapshot()
    summary = {
        "path": " -> ".join(path),
        "rewrites": path.count("transform_query"),
        "generations": path.count("generate"),
        "llm_calls": calls_after - calls_before,
        "tokens": tokens_after - tokens_before,
        "latency_s": round(time.time() - started, 1),
        "finished": finished,
    }
    return state, summary


def show_result(state, summary):
    print("=" * 100)
    print("PATH:", summary["path"])
    print(f"query rewrites: {summary['rewrites']} | generations: {summary['generations']} | "
          f"LLM calls: {summary['llm_calls']} | tokens: {summary['tokens']:,} | "
          f"time: {summary['latency_s']} s | finished: {summary['finished']}")
    print("=" * 100)
    if not summary["finished"]:
        print("The run was stopped, so there is no final answer. The last recommendation it wrote "
              "(not accepted by the checks) was:\n")
    print(state.get("generation", "(no recommendation was produced)"))


scenario_results = {}  # scenario -> (question, final state, run summary)

In [16]:
# Scenario A: Clean path - well-covered pricing question
inputs = {
    "question": "I sell Puma running sneakers that cost me $1800. My competitor prices at $2799 "
                "and I want a 30% margin. What price should I set?"
}
state_a, summary_a = run_agent(app, inputs["question"])
scenario_results["A"] = (inputs["question"], state_a, summary_a)
show_result(state_a, summary_a)

---RETRIEVE---
Node 'retrieve':

---

---CHECK DOCUMENT RELEVANCE TO PRICING QUESTION---
---GRADE: PRICING DOCUMENT RELEVANT---      [decision / Footwear]
---GRADE: PRICING DOCUMENT RELEVANT---      [decision / Footwear]
---GRADE: PRICING DOCUMENT RELEVANT---      [guideline / General]
---GRADE: PRICING DOCUMENT RELEVANT---      [margin / Footwear]
---ASSESS GRADED PRICING DOCUMENTS---
---DECISION: GENERATE---
Node 'grade_documents':

---

---GENERATE PRICE RECOMMENDATION---
---CHECK HALLUCINATIONS IN PRICE RECOMMENDATION---
---DECISION: RECOMMENDATION IS GROUNDED IN PRICING DATA---
---GRADE RECOMMENDATION VS QUESTION---
---DECISION: RECOMMENDATION ANSWERS THE PRICING QUESTION---
Node 'generate':

---

PATH: retrieve -> grade_documents -> generate
query rewrites: 0 | generations: 1 | LLM calls: 7 | tokens: 5,676 | time: 3.5 s | finished: True
**Recommended price:** $2,570  

**Margin calculation:**  
\[
\text{Margin} = \frac{\text{price} - \text{cost}}{\text{price}} = \frac{2,570 - 1,8

In [17]:
# Scenario B: Vague merchant question - the kind the query rewriter is for
inputs = {"question": "my sneakers aren't selling, should I drop the price?"}

state_b, summary_b = run_agent(app, inputs["question"])
scenario_results["B"] = (inputs["question"], state_b, summary_b)
show_result(state_b, summary_b)
print(f"\ntransform_query fired {summary_b['rewrites']} time(s).")

---RETRIEVE---
Node 'retrieve':

---

---CHECK DOCUMENT RELEVANCE TO PRICING QUESTION---
---GRADE: PRICING DOCUMENT RELEVANT---      [decision / Footwear]
---GRADE: PRICING DOCUMENT RELEVANT---      [decision / Footwear]
---GRADE: PRICING DOCUMENT NOT RELEVANT---  [decision / Apparel]
---GRADE: PRICING DOCUMENT NOT RELEVANT---  [decision / Apparel]
---ASSESS GRADED PRICING DOCUMENTS---
---DECISION: GENERATE---
Node 'grade_documents':

---

---GENERATE PRICE RECOMMENDATION---
---CHECK HALLUCINATIONS IN PRICE RECOMMENDATION---
---DECISION: RECOMMENDATION IS GROUNDED IN PRICING DATA---
---GRADE RECOMMENDATION VS QUESTION---
---DECISION: RECOMMENDATION ANSWERS THE PRICING QUESTION---
Node 'generate':

---

PATH: retrieve -> grade_documents -> generate
query rewrites: 0 | generations: 1 | LLM calls: 7 | tokens: 5,996 | time: 3.8 s | finished: True
The knowledge base does not contain data for this specific sneaker — you haven’t provided the cost of your product or the competitor’s price, whi

In [18]:
# Scenario C: Out-of-scope category - should NOT hallucinate
inputs = {
    "question": "What margin should I target on organic grocery produce costing $3 per unit?"
}

state_c, summary_c = run_agent(app, inputs["question"])
scenario_results["C"] = (inputs["question"], state_c, summary_c)
show_result(state_c, summary_c)

# Did the agent admit the knowledge base has no Grocery data, or did it invent one?
answer_c = state_c.get("generation", "")
print("\nAdmits the data gap:", "does not contain data" in answer_c.lower())
print("Percentages in the answer:", re.findall(r"\d+(?:\.\d+)?\s?%", answer_c) or "none")

---RETRIEVE---
Node 'retrieve':

---

---CHECK DOCUMENT RELEVANCE TO PRICING QUESTION---
---GRADE: PRICING DOCUMENT RELEVANT---      [guideline / General]
---GRADE: PRICING DOCUMENT NOT RELEVANT---  [margin / Electronics]
---GRADE: PRICING DOCUMENT NOT RELEVANT---  [margin / Apparel]
---GRADE: PRICING DOCUMENT NOT RELEVANT---  [margin / Luxury]
---ASSESS GRADED PRICING DOCUMENTS---
---DECISION: GENERATE---
Node 'grade_documents':

---

---GENERATE PRICE RECOMMENDATION---
---CHECK HALLUCINATIONS IN PRICE RECOMMENDATION---
---DECISION: RECOMMENDATION NOT GROUNDED, RE-TRY---
Node 'generate':

---

---GENERATE PRICE RECOMMENDATION---
---CHECK HALLUCINATIONS IN PRICE RECOMMENDATION---
---DECISION: RECOMMENDATION IS GROUNDED IN PRICING DATA---
---GRADE RECOMMENDATION VS QUESTION---
---DECISION: RECOMMENDATION ANSWERS THE PRICING QUESTION---
Node 'generate':

---

PATH: retrieve -> grade_documents -> generate -> generate
query rewrites: 0 | generations: 2 | LLM calls: 9 | tokens: 8,089 | time

**Scenario D: the rewrite loop in action.** In Scenarios A–C at least one retrieved document
always passes the (deliberately lenient) relevance grader, so `transform_query` never has to fire.
This question was found by testing a few casual phrasings: for "phone case", the retriever returns
the smartphone and headphone decisions plus Luxury and Footwear entries, and the grader rejects all
four. That sends the graph round the `transform_query → retrieve` loop.

In [19]:
# Scenario D: every retrieved document fails grading, so the query has to be rewritten
inputs = {"question": "what should I charge for a phone case that costs me $4?"}

state_d, summary_d = run_agent(app, inputs["question"])
scenario_results["D"] = (inputs["question"], state_d, summary_d)
show_result(state_d, summary_d)
print(f"\ntransform_query fired {summary_d['rewrites']} time(s).")

---RETRIEVE---
Node 'retrieve':

---

---CHECK DOCUMENT RELEVANCE TO PRICING QUESTION---
---GRADE: PRICING DOCUMENT NOT RELEVANT---  [decision / Electronics]
---GRADE: PRICING DOCUMENT NOT RELEVANT---  [decision / Luxury]
---GRADE: PRICING DOCUMENT NOT RELEVANT---  [decision / Electronics]
---GRADE: PRICING DOCUMENT NOT RELEVANT---  [decision / Footwear]
---ASSESS GRADED PRICING DOCUMENTS---
---DECISION: NO RELEVANT PRICING DOCUMENTS, TRANSFORM QUERY---
Node 'grade_documents':

---

---TRANSFORM QUERY---
   new question: What should be the optimal selling price for a phone case (Electronics accessory) that costs $4, considering price elasticity benchmarks, desired gross margin, and relevant pricing guidelines?
Node 'transform_query':

---

---RETRIEVE---
Node 'retrieve':

---

---CHECK DOCUMENT RELEVANCE TO PRICING QUESTION---
---GRADE: PRICING DOCUMENT RELEVANT---      [decision / Electronics]
---GRADE: PRICING DOCUMENT RELEVANT---      [elasticity / Electronics]
---GRADE: PRICING DOC

**Observations on the scenarios**

**A, clean path.**
- Path: `retrieve → grade_documents → generate → END` in 7 LLM calls (4 document grades, 1 answer, 2 checks).
- Answer: $2,570 for about a 30% margin, about 8% below the $2,799 competitor, citing the Puma decision.
- Small slip the graders don't catch: (2,570 − 1,800) / 2,570 = 29.96%, just under the 30% target. The exact price is $2,571.43, so it should round up. Exact arithmetic is a job for code, like stage 4's tools.

**B, vague question.**
- `transform_query` fired 0 times. Grading dropped the two Apparel decisions and kept the two Footwear ones.
- The answer says what is missing (the merchant's cost and the competitor's price) and gives the footwear rule: match the competitor for medium elasticity, about 7% below for high.
- Slip: it says both past decisions aimed for about a 30% margin, but the Nike decision was 33%.
- The rewrite branch only fires when *every* document fails grading, and the relevance grader is deliberately lenient.

**C, out of scope.**
- No invented grocery margin: the answer says the knowledge base has no data for the category, explains the margin floor rule, and treats the guideline's 25% as an example.
- The path shows a rejection: the first answer was judged *not grounded* and rewritten (9 calls). But the rejected answer was the more cautious one ("apply the same formula with whatever minimum margin you deem acceptable"). The accepted one says "set a price of at least $4.00, which yields a 25% margin". The check doesn't always prefer the better answer.

**D, the rewrite loop.**
- All four documents from the first retrieval were rejected. The rewriter labelled the product an Electronics accessory, and the new retrieval found the Electronics benchmark and two Electronics decisions.
- The answer applies the benchmark's rule: price at or just below the leading competitor (12 calls).
- It illustrates the rule with "if the competitor charges $9". That is labelled as an example, but the number is made up, and the grader passed it.

**What earlier runs showed:**
- **The generator borrowed a number.** With the generator at low reasoning effort and a weaker "no data" rule, every grocery answer, from this agent and from plain RAG alike, was "$4.00, a 25% margin", and every grader passed it. That is why the generator now checks for category data first.
- **The plain regenerate loop spun.** Another run rejected its grocery answer 8 times in a row (31 calls) before one passed, because a plain regenerate sends the same prompt again. That run is the reason for section 10.

## 10. A retry limit

The graph in section 8 can loop **forever**: if the groundedness grader never says yes, `generate → not supported → generate` cycles indefinitely, making API calls on every turn. The same goes for a question whose documents never pass grading.

A counter, `retries`, goes up each time the agent writes an answer or rewrites the question. After 3 attempts, a failed check stops the run, and the answer is returned with a **low-confidence warning**.

A rewrite after a failed groundedness check is also told that its previous answer was rejected, and is shown that answer, so it has a reason to write something different.

The cell then runs Scenario C on the capped graph, plus a worst case with a fake grader that never says yes.

In [20]:
# 10. Retry limit
#
# One counter, `retries`, goes up every time the agent writes an answer or rewrites the query, so
# both loops (regenerate and re-retrieve) share one budget. Three design choices:
#   - the last answer is still graded; the agent stops only when it FAILS a check after the cap,
#     and a separate node then adds the low-confidence warning (`generate` cannot know in advance
#     whether its answer will pass);
#   - a regeneration is told that its previous answer was rejected, and is shown that answer.
#     A plain regenerate sends the exact same prompt again, and at temperature 0 the model mostly
#     writes the same answer again (the repeated runs in section 11 show near-identical answers to
#     the same question);
#   - transform_query clears the old answer, so a regeneration can tell the two loops apart.
MAX_RETRIES = 3
LOW_CONFIDENCE_WARNING = (
    "LOW CONFIDENCE: this recommendation did not pass the agent's own checks within "
    f"{MAX_RETRIES} attempts. Verify every number before using it.\n\n"
)
RETRY_NOTE = """

A previous answer to this question (below) was rejected because some of its numbers or claims are
not supported by the retrieved context. Write a new answer. Keep a number only if the context states
it for this product's category, or it follows from figures the merchant gave; leave out anything
else and say what data is missing instead.

Rejected answer:
{previous}"""


class BoundedGraphState(GraphState, total=False):
    retries: int


def generate_bounded(state):
    retries = state.get("retries", 0) + 1
    previous = state.get("generation")
    if not previous:  # first answer for these documents
        update = generate(state)
    else:             # we came back from "not supported": regenerate with feedback
        print("---REGENERATE PRICE RECOMMENDATION (previous answer rejected)---")
        question = state["question"] + RETRY_NOTE.format(previous=previous)
        update = {"generation": rag_chain.invoke({"context": format_docs(state["documents"]),
                                                  "question": question})}
    print(f"   (attempt {retries} of {MAX_RETRIES})")
    return {**update, "retries": retries}


def transform_query_bounded(state):
    retries = state.get("retries", 0) + 1
    update = transform_query(state)
    print(f"   (attempt {retries} of {MAX_RETRIES})")
    return {**update, "generation": "", "retries": retries}


def decide_to_generate_bounded(state):
    if not state["documents"] and state.get("retries", 0) >= MAX_RETRIES:
        print("---RETRY CAP REACHED: ANSWER WITHOUT DOCUMENTS (THE KNOWLEDGE BASE HAS NO DATA)---")
        return "generate"
    return decide_to_generate(state)


def grade_generation_bounded(state):
    decision = grade_generation_v_documents_and_question(state)
    if decision != "useful" and state.get("retries", 0) >= MAX_RETRIES:
        print("---RETRY CAP REACHED: STOP AND FLAG THE ANSWER---")
        return "give up"
    return decision


def flag_low_confidence(state):
    print("---FLAG LOW-CONFIDENCE RECOMMENDATION---")
    return {"generation": LOW_CONFIDENCE_WARNING + state["generation"]}


bounded = StateGraph(BoundedGraphState)
bounded.add_node("retrieve", retrieve)
bounded.add_node("grade_documents", grade_documents)
bounded.add_node("generate", generate_bounded)
bounded.add_node("transform_query", transform_query_bounded)
bounded.add_node("flag_low_confidence", flag_low_confidence)
bounded.add_edge(START, "retrieve")
bounded.add_edge("retrieve", "grade_documents")
bounded.add_edge("transform_query", "retrieve")
bounded.add_conditional_edges("grade_documents", decide_to_generate_bounded,
                              {"transform_query": "transform_query", "generate": "generate"})
bounded.add_conditional_edges("generate", grade_generation_bounded,
                              {"not supported": "generate", "not useful": "transform_query",
                               "useful": END, "give up": "flag_low_confidence"})
bounded.add_edge("flag_low_confidence", END)
app_bounded = bounded.compile()
print_edges(app_bounded)

# Re-run Scenario C on the bounded graph
print()
question_c = scenario_results["C"][0]
state_cb, summary_cb = run_agent(app_bounded, question_c)
show_result(state_cb, summary_cb)
print(f"\nretries used: {state_cb.get('retries')}")

# Worst case: with a fake groundedness grader that never says yes, the run must still end.
real_graders = hallucination_grader, answer_grader
always_no = RunnableLambda(lambda _: GradeHallucinations(binary_score="no"))
hallucination_grader = always_no
try:
    print("\nWorst case: a groundedness grader that never says yes")
    state_w, summary_w = run_agent(app_bounded, "I sell Puma running sneakers that cost me $1800. What price?",
                                   verbose=False)
    print(f"  path: {summary_w['path']}")
    print(f"  finished: {summary_w['finished']} | retries: {state_w.get('retries')} | "
          f"answer starts with the warning: {state_w['generation'].startswith('LOW CONFIDENCE')}")
finally:
    hallucination_grader, answer_grader = real_graders

  __start__            ------> retrieve
  generate             ..useful..> __end__
  generate             ..give up..> flag_low_confidence
  generate             ..not supported..> generate
  generate             ..not useful..> transform_query
  grade_documents      ..generate..> generate
  grade_documents      ..transform_query..> transform_query
  retrieve             ------> grade_documents
  transform_query      ------> retrieve
  flag_low_confidence  ------> __end__

---RETRIEVE---
Node 'retrieve':

---

---CHECK DOCUMENT RELEVANCE TO PRICING QUESTION---
---GRADE: PRICING DOCUMENT RELEVANT---      [guideline / General]
---GRADE: PRICING DOCUMENT NOT RELEVANT---  [margin / Electronics]
---GRADE: PRICING DOCUMENT NOT RELEVANT---  [margin / Apparel]
---GRADE: PRICING DOCUMENT NOT RELEVANT---  [margin / Luxury]
---ASSESS GRADED PRICING DOCUMENTS---
---DECISION: GENERATE---
Node 'grade_documents':

---

---GENERATE PRICE RECOMMENDATION---
   (attempt 1 of 3)
---CHECK HALLUCINATIONS IN

**What this shows**

- **On the capped graph, Scenario C passed at the first attempt this time** (7 calls).
- **The worst case still ends.** With a fake grader that never says yes, the agent writes the answer 3 times, the last two with feedback, then stops and flags it.
- **A retry counter is better than `recursion_limit` for a merchant-facing tool:**
  - it ends with an answer and a visible warning, whereas `recursion_limit` ends with an exception and no answer;
  - it counts real attempts (answers written plus questions rewritten), not graph steps.

  `recursion_limit` stays as a backstop.
- **The cap also fired in section 11.** One grocery run in the comparison reached it and was flagged LOW CONFIDENCE, although its text was reasonable: it asks for the merchant's minimum margin and uses 25% only as an example. That is one more sign the checks are noisy.

## 11. Self-RAG vs naive RAG

The baseline is a linear pipeline with the same retriever, prompt and model, and no grading or loops: `retriever | format_docs → pricing_prompt → llm`. Both agents answer every scenario 3 times, because answers vary from run to run.

The **groundedness grader from section 4 is the judge** of whether each answer is hallucinated. The table reports LLM calls, tokens and latency per question, so the cost of the extra checks is visible next to what they buy.

In [21]:
# 11. Self-RAG vs naive RAG
import time

# Step 1 - Build the naive (linear, ungraded) RAG chain: same retriever, prompt and model
naive_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | pricing_prompt | llm | StrOutputParser() | RunnableLambda(one_copy)
)

# Step 2 - Run both agents on all three scenarios, timing each. Answers vary from run to run, so
# each scenario runs N_RUNS times per agent.
N_RUNS = 3
SCENARIOS = {"A: clean path": scenario_results["A"][0],
             "B: vague question": scenario_results["B"][0],
             "C: out of scope (Grocery)": scenario_results["C"][0],
             "D: rewrite needed": scenario_results["D"][0]}


def judge(documents, question, answer):
    """Step 4: section 4's groundedness grader as the judge. Returns 'y' if the answer is NOT grounded.
    Judge calls are not counted in the agents' cost."""
    calls, tokens = COUNTER.calls, COUNTER.tokens
    score = hallucination_grader.invoke(
        {"documents": format_docs(documents), "question": question, "generation": answer}).binary_score
    COUNTER.calls, COUNTER.tokens = calls, tokens
    return "n" if score == "yes" else "y"


runs = []
for scenario, question in SCENARIOS.items():
    for run in range(1, N_RUNS + 1):
        # Naive RAG: one retrieval, one LLM call, no checks
        calls0, tokens0 = COUNTER.snapshot()
        started = time.time()
        naive_answer = naive_rag_chain.invoke(question)
        naive_latency = time.time() - started
        naive_calls, naive_tokens = COUNTER.calls - calls0, COUNTER.tokens - tokens0
        naive_docs = retriever.invoke(question)

        # Self-RAG: the bounded graph from section 10
        state, summary = run_agent(app_bounded, question, verbose=False)
        selfrag_answer = state.get("generation", "")

        runs.append({
            "scenario": scenario, "run": run,
            "naive_answer": naive_answer, "selfrag_answer": selfrag_answer,
            "naive_hallucinated": judge(naive_docs, question, naive_answer),
            # judged against the documents the answer was finally written from
            "selfrag_hallucinated": judge(state.get("documents", []), state.get("question", question),
                                          selfrag_answer),
            "naive_llm_calls": naive_calls, "selfrag_llm_calls": summary["llm_calls"],
            "naive_tokens": naive_tokens, "selfrag_tokens": summary["tokens"],
            "naive_latency_s": naive_latency, "selfrag_latency_s": summary["latency_s"],
            "selfrag_path": summary["path"],
        })
        print(f"{scenario} run {run}: naive {naive_latency:.1f} s | Self-RAG {summary['latency_s']} s, "
              f"{summary['llm_calls']} calls, path: {summary['path']}")

runs_df = pd.DataFrame(runs)


# Step 3 - The comparison table: one row per scenario
def snippet(text, width=90):
    text = " ".join(text.split())
    return text if len(text) <= width else text[:width] + "..."


def count_y(flags):
    return f"{(flags == 'y').sum()}/{len(flags)}"


first_run = runs_df[runs_df["run"] == 1].set_index("scenario")
table = runs_df.groupby("scenario", sort=False).agg(
    naive_hallucinated=("naive_hallucinated", count_y),
    selfrag_hallucinated=("selfrag_hallucinated", count_y),
    naive_llm_calls=("naive_llm_calls", "mean"),
    selfrag_llm_calls=("selfrag_llm_calls", "mean"),
    naive_latency_s=("naive_latency_s", "mean"),
    selfrag_latency_s=("selfrag_latency_s", "mean"),
)
table.insert(0, "naive_answer_snippet", first_run["naive_answer"].map(snippet))
table.insert(1, "selfrag_answer_snippet", first_run["selfrag_answer"].map(snippet))

# Step 5 - Print the comparison and the cost/quality trade-off
with pd.option_context("display.max_colwidth", 95, "display.width", 250, "display.precision", 1):
    print("\nHallucinated = judged NOT grounded by the section 4 grader (count over the runs).")
    print("llm_calls and latency_s are averages per run.\n")
    print(table.reset_index().to_string(index=False))

naive_calls, selfrag_calls = runs_df["naive_llm_calls"].mean(), runs_df["selfrag_llm_calls"].mean()
naive_time, selfrag_time = runs_df["naive_latency_s"].mean(), runs_df["selfrag_latency_s"].mean()
naive_tok, selfrag_tok = runs_df["naive_tokens"].mean(), runs_df["selfrag_tokens"].mean()
print(f"\nCost per question (average over {len(runs_df)} runs per agent):")
print(f"  LLM calls: naive {naive_calls:.1f} vs Self-RAG {selfrag_calls:.1f} ({selfrag_calls / naive_calls:.1f}x)")
print(f"  tokens:    naive {naive_tok:,.0f} vs Self-RAG {selfrag_tok:,.0f} ({selfrag_tok / naive_tok:.1f}x)")
print(f"  latency:   naive {naive_time:.1f} s vs Self-RAG {selfrag_time:.1f} s ({selfrag_time / naive_time:.1f}x)")
print(f"Answers judged not grounded: naive {(runs_df['naive_hallucinated'] == 'y').sum()}/{len(runs_df)}, "
      f"Self-RAG {(runs_df['selfrag_hallucinated'] == 'y').sum()}/{len(runs_df)}")

---RETRIEVE---
---CHECK DOCUMENT RELEVANCE TO PRICING QUESTION---
---GRADE: PRICING DOCUMENT RELEVANT---      [decision / Footwear]
---GRADE: PRICING DOCUMENT RELEVANT---      [decision / Footwear]
---GRADE: PRICING DOCUMENT RELEVANT---      [guideline / General]
---GRADE: PRICING DOCUMENT RELEVANT---      [margin / Footwear]
---ASSESS GRADED PRICING DOCUMENTS---
---DECISION: GENERATE---
---GENERATE PRICE RECOMMENDATION---
   (attempt 1 of 3)
---CHECK HALLUCINATIONS IN PRICE RECOMMENDATION---
---DECISION: RECOMMENDATION IS GROUNDED IN PRICING DATA---
---GRADE RECOMMENDATION VS QUESTION---
---DECISION: RECOMMENDATION ANSWERS THE PRICING QUESTION---
A: clean path run 1: naive 0.6 s | Self-RAG 3.6 s, 7 calls, path: retrieve -> grade_documents -> generate
---RETRIEVE---
---CHECK DOCUMENT RELEVANCE TO PRICING QUESTION---
---GRADE: PRICING DOCUMENT RELEVANT---      [decision / Footwear]
---GRADE: PRICING DOCUMENT RELEVANT---      [decision / Footwear]
---GRADE: PRICING DOCUMENT RELEVANT---  

In [22]:
# Every answer in full, so the judge's verdicts can be checked by reading them
for row in runs:
    print("=" * 100)
    print(f"{row['scenario']} | run {row['run']}")
    print(f"--- NAIVE (judged hallucinated: {row['naive_hallucinated']}) ---")
    print(row["naive_answer"])
    print(f"--- SELF-RAG (judged hallucinated: {row['selfrag_hallucinated']}; path: {row['selfrag_path']}) ---")
    print(row["selfrag_answer"])

A: clean path | run 1
--- NAIVE (judged hallucinated: n) ---
**Recommended price:** $2,570  

**Margin calculation:**  
\[
\text{Margin} = \frac{\text{price} - \text{cost}}{\text{price}} = \frac{2,570 - 1,800}{2,570} = \frac{770}{2,570} \approx 30\%
\]

**Evidence:** This recommendation follows the **past pricing decision for Puma Running Sneakers** (Footwear) where, with the same $1,800 cost and a competitor price of $2,799, we set $2,599 to achieve a 31 % margin and successfully under‑cut the competitor by 7 % (high‑elasticity case). Targeting a 30 % margin yields a price of $2,570, still about 8 % below the competitor, which aligns with the same elasticity reasoning.  

No conflict with a target‑margin price arises; the suggested $2,570 meets the 30 % margin goal while preserving the competitive advantage demonstrated in the prior decision.
--- SELF-RAG (judged hallucinated: n; path: retrieve -> grade_documents -> generate) ---
**Recommended price:** $2,570  

**Margin calculation:*

**Observations on the comparison**

**Cost per question**, averaged over 12 runs per agent:

| | Naive RAG | Self-RAG | Ratio |
|---|---|---|---|
| LLM calls | 1 | 9.3 | 9.3× |
| Tokens | 1.1k | 7.7k | 6.8× |
| Latency | 0.7 s | 5.2 s | 7.6× |

The clean path costs 7 calls; a rewrite costs 12-14.

**The judge says naive 4/12 and Self-RAG 1/12, but read those numbers with care:**
- **They are circular:** every Self-RAG answer had already passed this same grader inside the loop.
- **They are noisy:**
  - the three naive grocery answers are word for word the same, and got *no, yes, yes*;
  - Self-RAG's phone-case answer in run 1 is word for word the one this grader accepted in section 9, and this time it was judged hallucinated.

**Reading all 24 answers:**
- **D (phone case) is the clear difference.**
  - Naive RAG invented "30-40% for high-elasticity items" (runs 1 and 2), or borrowed the smartphone's 11% margin to suggest **$4.49 for a $4 case** (run 3). The judge rated run 3 *not* hallucinated.
  - Self-RAG rejected the off-topic documents, rewrote the question, and answered from the Electronics benchmark all 3 times.
- **A and B:** both agents gave the same quality of answer.
- **C:** both admit the data gap. Naive RAG then suggests a made-up "e.g. 20%" ($3.75 floor); Self-RAG uses the guideline's example 25% ($4.00), and was once flagged low confidence.

**Is the trade-off worth it?** For a merchant-facing pricing tool, yes: a few seconds and about 7× the tokens per question is small next to a wrong price, and D shows the kind of error it prevents. But much of the protection in this run comes from the strict generator prompt, and the graders disagree with themselves on identical text. Numbers should also be checked by code, as in stage 4's tools and price check. For low-stakes text, such as product descriptions, plain RAG with a good prompt and spot checks is enough.

## Summary

**What works**
- Relevance grading keeps off-topic categories out of the context.
- The rewrite loop recovers from a failed retrieval (D).
- Questions about an uncovered category get "the knowledge base does not contain data for this category" instead of a borrowed margin (C).
- Every loop is bounded, and an answer that fails its checks is returned with a visible warning.

**What doesn't work yet**
- **LLM graders are noisy on subtle cases.** A borrowed example number passed 3 times out of 3, identical answers got different verdicts, and a cautious answer was rejected while a bolder one passed.
- **The graders don't catch small arithmetic slips** (29.96% sold as 30%) or made-up illustrative numbers.
- **The quality comes partly from the prompt.** Much of what keeps the answers honest is the generator prompt, not the loop.

**Next stage:** structured output with the price as a number, checked in code, and stage 4's tools for the arithmetic, with these graders kept as a second line of defence.